# NZRP workflow

This notebook is the reproducibility wrapper for the repository.

Current state:
- Part 1 implemented: compile the simulator and run batches of simulations
- Part 2 placeholder: convolution wrappers
- Part 3 placeholder: analysis, fits, and plots

## Part 1: Simulation wrapper

In [1]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import time
from dataclasses import dataclass

try:
    from scipy.optimize import curve_fit
except ImportError:
    print("Scipy.optimize is not installed. Please install it to proceed.")


try:
    import pandas as pd
except ImportError:
    print("Pandas is not installed. Please install it to proceed.")


In [2]:
REPO = Path.cwd()
CODE_DIR = REPO / "code"
CONV_DIR = REPO / "conv"
PARTIALRANGE_CONV_DIR = REPO / "partialrange_conv"
AGG_DIR = REPO / "aggregate"
FULLRANGE_CONV_DIR = REPO / "fullrange_conv"
RES_DIR = REPO / "res"
CONV_RES_DIR = REPO / "conv_res"

if not (CODE_DIR / "main.cpp").exists():
    raise FileNotFoundError(f"Notebook must be run from the repo root. I looked for {CODE_DIR / 'main.cpp'}")

AGG_DIR.mkdir(exist_ok=True)
FULLRANGE_CONV_DIR.mkdir(exist_ok=True)
PARTIALRANGE_CONV_DIR.mkdir(exist_ok=True)
RES_DIR.mkdir(exist_ok=True)
CONV_RES_DIR.mkdir(exist_ok=True)

print(f"Agg dir  : {AGG_DIR}")
print(f"Full conv: {FULLRANGE_CONV_DIR}")
print(f"Partial  : {PARTIALRANGE_CONV_DIR}")
print(f"Repo root: {REPO}")
print(f"Code dir : {CODE_DIR}")
print(f"Res dir  : {RES_DIR}")

Agg dir  : /home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/aggregate
Full conv: /home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/fullrange_conv
Partial  : /home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/partialrange_conv
Repo root: /home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo
Code dir : /home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/code
Res dir  : /home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/res


In [3]:
Ls = [32, 64, 127, 256]
Ts = [70, 40, 20, 5]
run_numbers = [10, 8, 6, 4]


# Ls = [64, 90, 128, 181, 256]
# Ts = [100, 50, 50, 25, 25]
# run_numbers = [7, 7, 7, 7, 7]


compile_with_optimization = True
compile_with_warnings = False
compile_with_debug = False
compiler_override = None         # e.g. 'g++-10' or 'clang++' if the default g++ is too old for C++20

sleep_between_runs = 0.0
stop_on_error = True

if len(Ls) != len(Ts) or len(Ls) != len(run_numbers):
    raise ValueError("Ls, Ts, and run_numbers must have the same length")

print("System sizes :", Ls)
print("Trials       :", Ts)
print("Run numbers  :", run_numbers)

System sizes : [32, 64, 127, 256]
Trials       : [70, 40, 20, 5]
Run numbers  : [10, 8, 6, 4]


In [4]:
@dataclass
class RunRecord:
    L: int
    T: int
    num: int
    returncode: int
    wall_time: float


def run_cmd(cmd, cwd, check=True):
    print("$", " ".join(str(x) for x in cmd))
    t0 = time.time()
    proc = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    dt = time.time() - t0
    if proc.stdout.strip():
        print(proc.stdout)
    if proc.stderr.strip():
        print(proc.stderr, file=sys.stderr)
    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {proc.returncode}: {' '.join(cmd)}")
    return proc, dt


def find_make():
    for name in ["make", "mingw32-make"]:
        path = shutil.which(name)
        if path is not None:
            return path
    return None


def make_command(*extra_args):
    make = find_make()
    if make is None:
        raise RuntimeError("Could not find make or mingw32-make in PATH")
    cmd = [make, "-f", "makefile"]
    if compiler_override:
        cmd.append(f"CXX={compiler_override}")
    cmd.extend(extra_args)
    return cmd

def simulator_exe_path():
    return CODE_DIR / "rp.exe"

def compile_simulator(optimize=True, warnings=False, debug=False):
    cmd = make_command("clean")
    run_cmd(cmd, cwd=CODE_DIR, check=True)

    cmd = make_command()
    if optimize:
        cmd.append("o=1")
    if warnings:
        cmd.append("w=1")
    if debug:
        cmd.append("g=1")
    try:
        run_cmd(cmd, cwd=CODE_DIR, check=True)
    except RuntimeError as exc:
        hint = " If the compiler is too old for -std=c++20, set compiler_override = 'g++-10' (or another C++20-capable compiler) in the configuration cell."
        raise RuntimeError(str(exc) + hint) from exc

    exe = simulator_exe_path()
    if not exe.exists():
        raise FileNotFoundError(f"Compilation ended but {exe} does not exist")
    return exe


def expected_simulation_files(L, T, num):
    stems = [
        "LOG",
        "Smax",
        "Searches",
        "TimePerBond",
        "Events",
        "Operations",
        "EventTimes",
        "PERFLOG",
    ]
    return {stem: RES_DIR / f"{stem}_L{L}_T{T}_num{num}.txt" for stem in stems}


def run_one_simulation(L, T, num, check_outputs=True):
    exe = simulator_exe_path()
    if not exe.exists():
        raise FileNotFoundError(f"Simulator executable not found: {exe}")

    proc, dt = run_cmd([str(exe), str(L), str(T), str(num)], cwd=CODE_DIR, check=False)
    if proc.returncode != 0 and stop_on_error:
        raise RuntimeError(f"Simulation failed for L={L}, T={T}, num={num}")

    if check_outputs:
        expected = expected_simulation_files(L, T, num)
        missing = [name for name, path in expected.items() if not path.exists()]
        if missing and stop_on_error:
            raise FileNotFoundError(f"Missing output files for L={L}, T={T}, num={num}: {missing}")

    return RunRecord(L=L, T=T, num=num, returncode=proc.returncode, wall_time=dt)


def run_simulation_batch(Ls, Ts, run_numbers):
    records = []
    for i, pair in enumerate(zip(Ls, Ts)):
        L, T = pair
        for num in range(run_numbers[i]):
            print(f"Running simulation: L={L}, T={T}, num={num}")
            rec = run_one_simulation(L, T, num)
            records.append(rec)
            if sleep_between_runs > 0:
                time.sleep(sleep_between_runs)
    return records


def simulation_status_table(Ls, Ts, run_numbers):
    rows = []
    for i, pair in enumerate(zip(Ls, Ts)):
        L, T = pair
        for num in range(run_numbers[i]):
            files = expected_simulation_files(L, T, num)
            row = {"L": L, "T": T, "num": num}
            for name, path in files.items():
                row[name] = path.exists()
            rows.append(row)

    if pd is not None:
        return pd.DataFrame(rows)
    return rows

In [5]:
simulator_exe = compile_simulator(
    optimize=compile_with_optimization,
    warnings=compile_with_warnings,
    debug=compile_with_debug,
)

print("Built:", simulator_exe)

$ /usr/bin/make -f makefile clean
rm -f main.o basic_functions.o percolation.o pebble_game.o measurements.o rp.exe

$ /usr/bin/make -f makefile o=1
g++ -std=c++20 -O3 -march=native -flto -ffast-math -c main.cpp -o main.o
g++ -std=c++20 -O3 -march=native -flto -ffast-math -c basic_functions.cpp -o basic_functions.o
g++ -std=c++20 -O3 -march=native -flto -ffast-math -c percolation.cpp -o percolation.o
g++ -std=c++20 -O3 -march=native -flto -ffast-math -c pebble_game.cpp -o pebble_game.o
g++ -std=c++20 -O3 -march=native -flto -ffast-math -c measurements.cpp -o measurements.o
g++ -std=c++20 -O3 -march=native -flto -ffast-math -o rp.exe main.o basic_functions.o percolation.o pebble_game.o measurements.o

Built: /home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/code/rp.exe


lto-wrapper: warning: using serial compilation of 2 LTRANS jobs



In [6]:
records = run_simulation_batch(Ls, Ts, run_numbers)

if pd is not None:
    pd.DataFrame([r.__dict__ for r in records])
else:
    records

Running simulation: L=32, T=70, num=0
$ /home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/code/rp.exe 32 70 0
Running simulation: L=32, T=70, num=1
$ /home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/code/rp.exe 32 70 1
Running simulation: L=32, T=70, num=2
$ /home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/code/rp.exe 32 70 2
Running simulation: L=32, T=70, num=3
$ /home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/code/rp.exe 32 70 3
Running simulation: L=32, T=70, num=4
$ /home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/code/rp.exe 32 70 4
Running simulation: L=32, T=70, num=5
$ /home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/code/rp.exe 32 70 5
Running simulation: L=32, T=70, num=6
$ /home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/code/rp.exe 32 70 6
Running simulation: L=32, T=70, num=7
$ /home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/code/rp.exe 32 70 7
Running simulation: L=32, T=70, num=8
$ /home/danimuzi/Desktop/R

In [7]:
simulation_status_table(Ls, Ts, run_numbers)

,L,T,num,LOG,Smax,Searches,TimePerBond,Events,Operations,EventTimes,PERFLOG
0,32,70,0,True,True,True,True,True,True,True,True
1,32,70,1,True,True,True,True,True,True,True,True
2,32,70,2,True,True,True,True,True,True,True,True
3,32,70,3,True,True,True,True,True,True,True,True
4,32,70,4,True,True,True,True,True,True,True,True
5,32,70,5,True,True,True,True,True,True,True,True
6,32,70,6,True,True,True,True,True,True,True,True
7,32,70,7,True,True,True,True,True,True,True,True
8,32,70,8,True,True,True,True,True,True,True,True
9,32,70,9,True,True,True,True,True,True,True,True


## Part 2: Aggregation wrapper

This section only runs the external aggregator and reads its output.

In [8]:
def read_aggregated_table(path):
    headers = []
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.startswith('#'):
                headers.append(line.rstrip())
                continue
            parts = line.split()
            if parts:
                rows.append([float(x) for x in parts])
    return headers, pd.DataFrame(rows)


In [9]:
def ensure_tool_built(tool_dir, exe_name):
    exe = tool_dir / exe_name
    need_build = not exe.exists()
    if not need_build:
        exe_mtime = exe.stat().st_mtime
        for src in list(tool_dir.glob('*.cpp')) + list(tool_dir.glob('*.h')) + [tool_dir / 'makefile']:
            if src.exists() and src.stat().st_mtime > exe_mtime:
                need_build = True
                break
    if not need_build:
        return exe
    
    proc, dt = run_cmd(make_command('o=1'), cwd=tool_dir, check=False)
    if proc.returncode != 0 or not exe.exists():
        hint = " If the compiler is too old for -std=c++20, set compiler_override = 'g++-10' (or another C++20-capable compiler) in the configuration cell."
        raise RuntimeError(f'Failed to build {exe_name} in {tool_dir}.' + hint)
    
    return exe


def aggregate_tool(stem, L, T, first_num, last_num, mode, col=None):
    exe = ensure_tool_built(AGG_DIR, 'aggregate.exe')
    cmd = [str(exe), str(L), str(T), str(first_num), str(last_num), stem, mode]
    if col is not None:
        cmd.append(str(col))
    proc, dt = run_cmd(cmd, cwd=AGG_DIR, check=False)
    if proc.returncode != 0 and stop_on_error:
        raise RuntimeError(f'Aggregation failed for {stem}, mode={mode}, L={L}, T={T}')
    return dt


def aggregated_path(stem, L, T, first_num, last_num, mode, col=None):
    real_mode = 'avg' if mode == 'curve' else mode
    base = f'{stem}_L{L}_T{T}_num{first_num}_to_{last_num}'
    if col is not None:
        base += f'_col{col}'
    return REPO / 'agg_res' / f'{base}_{real_mode}.txt'


def load_aggregated_table(path):
    return read_aggregated_table(path)


def fullrange_conv_tool(stem, L, T, first_num, last_num, mode, ncols, cols, model=None):
    exe = ensure_tool_built(FULLRANGE_CONV_DIR, 'fullrange_conv.exe')
    cmd = [str(exe), str(L), str(T), str(first_num), str(last_num), stem, mode, str(ncols)]
    cmd.extend(str(c) for c in cols)
    if model is not None:
        cmd.append(str(model))
    proc, dt = run_cmd(cmd, cwd=FULLRANGE_CONV_DIR, check=False)
    if proc.returncode != 0 and stop_on_error:
        raise RuntimeError(f'Full-range convolution failed for {stem}, mode={mode}, L={L}, T={T}')
    return dt


def fullrange_convolved_path(outstem, L, T, first_num, last_num, suffix='convoluted'):
    return CONV_RES_DIR / f'{outstem}_L{L}_T{T}_num{first_num}_to_{last_num}_{suffix}.txt'


def run_fullrange_convolution_batch(specs):
    outputs = {}
    for spec in specs:
        outstem = spec.get('model', spec['stem'])
        fullrange_conv_tool(**spec)
        outputs[(spec['stem'], spec['mode'], outstem, spec['L'], spec['T'])] = {
            'averaged': fullrange_convolved_path(outstem, spec['L'], spec['T'], spec['first_num'], spec['last_num'], suffix='averaged'),
            'convoluted': fullrange_convolved_path(outstem, spec['L'], spec['T'], spec['first_num'], spec['last_num'], suffix='convoluted'),
        }
    return outputs


def partialrange_conv_tool(L, T, first_num, last_num, stem, cols):
    exe = ensure_tool_built(PARTIALRANGE_CONV_DIR, 'partialrange_conv.exe')
    cmd = [str(exe), str(L), str(T), str(first_num), str(last_num), stem] + [str(c) for c in cols]
    proc, dt = run_cmd(cmd, cwd=PARTIALRANGE_CONV_DIR, check=False)
    if proc.returncode != 0 and stop_on_error:
        raise RuntimeError(f'Partial-range convolution failed for {stem}, L={L}, T={T}')
    return dt


def partialrange_convolved_path(stem, L, T, first_num, last_num, suffix='convoluted'):
    return CONV_RES_DIR / f'{stem}_L{L}_T{T}_num{first_num}_to_{last_num}_{suffix}.txt'


def run_partialrange_convolution_batch(specs):
    outputs = {}
    for spec in specs:
        partialrange_conv_tool(**spec)
        outputs[(spec['stem'], spec['L'], spec['T'])] = {
            'averaged': partialrange_convolved_path(spec['stem'], spec['L'], spec['T'], spec['first_num'], spec['last_num'], suffix='averaged'),
            'convoluted': partialrange_convolved_path(spec['stem'], spec['L'], spec['T'], spec['first_num'], spec['last_num'], suffix='convoluted'),
        }
    return outputs


def run_aggregation_batch(specs):
    outputs = {}
    for spec in specs:
        aggregate_tool(**spec)
        out = aggregated_path(spec['stem'], spec['L'], spec['T'], spec['first_num'], spec['last_num'], spec['mode'], spec.get('col'))
        outputs[(spec['stem'], spec['mode'], spec['L'], spec['T'], spec.get('col'))] = out
    return outputs


aggregation_specs = []
for i, pairs in enumerate(zip(Ls, Ts)):
    L, T = pairs
    aggregation_specs.extend([
        dict(stem='Searches', L=L, T=T, first_num=0, last_num=run_numbers[i], mode='curve'),
        dict(stem='TimePerBond', L=L, T=T, first_num=0, last_num=run_numbers[i], mode='curve'),
        dict(stem='Events', L=L, T=T, first_num=0, last_num=run_numbers[i], mode='curve'),
        dict(stem='Operations', L=L, T=T, first_num=0, last_num=run_numbers[i], mode='curve'),
        dict(stem='EventTimes', L=L, T=T, first_num=0, last_num=run_numbers[i], mode='curve'),
        dict(stem='CPSxy', L=L, T=T, first_num=0, last_num=run_numbers[i], mode='samples', col=0),
        dict(stem='RPSxy', L=L, T=T, first_num=0, last_num=run_numbers[i], mode='samples', col=0),
        dict(stem='WRAPS', L=L, T=T, first_num=0, last_num=run_numbers[i], mode='samples', col=4),
        dict(stem='WRAPS', L=L, T=T, first_num=0, last_num=run_numbers[i], mode='samples', col=5),
        dict(stem='PERFLOG', L=L, T=T, first_num=0, last_num=run_numbers[i], mode='samples', col=0),
        dict(stem='PERFLOG', L=L, T=T, first_num=0, last_num=run_numbers[i], mode='samples', col=5),
        dict(stem='PERFLOG', L=L, T=T, first_num=0, last_num=run_numbers[i], mode='samples', col=6),
        dict(stem='PERFLOG', L=L, T=T, first_num=0, last_num=run_numbers[i], mode='samples', col=7),
        dict(stem='PERFLOG', L=L, T=T, first_num=0, last_num=run_numbers[i], mode='samples', col=8),
        dict(stem='PERFLOG', L=L, T=T, first_num=0, last_num=run_numbers[i], mode='samples', col=9),
        dict(stem='PERFLOG', L=L, T=T, first_num=0, last_num=run_numbers[i], mode='samples', col=10),
    ])

aggregation_outputs = run_aggregation_batch(aggregation_specs)
aggregation_outputs



$ /home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/aggregate/aggregate.exe 32 70 0 10 Searches curve
$ /home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/aggregate/aggregate.exe 32 70 0 10 TimePerBond curve
$ /home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/aggregate/aggregate.exe 32 70 0 10 Events curve
$ /home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/aggregate/aggregate.exe 32 70 0 10 Operations curve
$ /home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/aggregate/aggregate.exe 32 70 0 10 EventTimes curve
$ /home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/aggregate/aggregate.exe 32 70 0 10 CPSxy samples 0
$ /home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/aggregate/aggregate.exe 32 70 0 10 RPSxy samples 0
$ /home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/aggregate/aggregate.exe 32 70 0 10 WRAPS samples 4
$ /home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/aggregate/aggregate.exe 32 70 0 10 WRAPS samples 5
$ /home/da

{('Searches',
  'curve',
  32,
  70,
  None): PosixPath('/home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/agg_res/Searches_L32_T70_num0_to_10_avg.txt'),
 ('TimePerBond',
  'curve',
  32,
  70,
  None): PosixPath('/home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/agg_res/TimePerBond_L32_T70_num0_to_10_avg.txt'),
 ('Events',
  'curve',
  32,
  70,
  None): PosixPath('/home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/agg_res/Events_L32_T70_num0_to_10_avg.txt'),
 ('Operations',
  'curve',
  32,
  70,
  None): PosixPath('/home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/agg_res/Operations_L32_T70_num0_to_10_avg.txt'),
 ('EventTimes',
  'curve',
  32,
  70,
  None): PosixPath('/home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/agg_res/EventTimes_L32_T70_num0_to_10_avg.txt'),
 ('CPSxy',
  'samples',
  32,
  70,
  0): PosixPath('/home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/agg_res/CPSxy_L32_T70_num0_to_10_col0_samples.txt'),
 ('RPSxy',
  'samples'

In [10]:
fullrange_conv_specs = []
for L, T in zip(Ls, Ts):
    fullrange_conv_specs.extend([
        dict(stem='WRAPS', L=L, T=T, first_num=0, last_num=run_numbers[i], mode='wrap', ncols=2, cols=[0, 1], model='CP'),
        dict(stem='WRAPS', L=L, T=T, first_num=0, last_num=run_numbers[i], mode='wrap', ncols=2, cols=[2, 3], model='RP'),
        dict(stem='Searches', L=L, T=T, first_num=0, last_num=run_numbers[i], mode='curve', ncols=1, cols=[0]),
        dict(stem='TimePerBond', L=L, T=T, first_num=0, last_num=run_numbers[i], mode='curve', ncols=1, cols=[0]),
        dict(stem='Operations', L=L, T=T, first_num=0, last_num=run_numbers[i], mode='curve', ncols=3, cols=[0, 1, 2]),
        dict(stem='Events', L=L, T=T, first_num=0, last_num=run_numbers[i], mode='curve', ncols=3, cols=[0, 1, 2]),
    ])

fullrange_outputs = {}
fullrange_outputs = run_fullrange_convolution_batch(fullrange_conv_specs)

partialrange_conv_specs = []
for i, pairs in enumerate(zip(Ls, Ts)):
    L, T = pairs
    partialrange_conv_specs.extend([
        dict(stem='COP', L=L, T=T, first_num=0, last_num=run_numbers[i], cols=[0, 1, 2]),
        dict(stem='ROP', L=L, T=T, first_num=0, last_num=run_numbers[i], cols=[0, 1, 2]),
    ])

partialrange_outputs = {}
partialrange_outputs = run_partialrange_convolution_batch(partialrange_conv_specs)
partialrange_outputs
fullrange_outputs


$ /home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/fullrange_conv/fullrange_conv.exe 32 70 0 4 WRAPS wrap 2 0 1 CP
$ /home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/fullrange_conv/fullrange_conv.exe 32 70 0 4 WRAPS wrap 2 2 3 RP
$ /home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/fullrange_conv/fullrange_conv.exe 32 70 0 4 Searches curve 1 0
$ /home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/fullrange_conv/fullrange_conv.exe 32 70 0 4 TimePerBond curve 1 0
$ /home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/fullrange_conv/fullrange_conv.exe 32 70 0 4 Operations curve 3 0 1 2
$ /home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/fullrange_conv/fullrange_conv.exe 32 70 0 4 Events curve 3 0 1 2
$ /home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/fullrange_conv/fullrange_conv.exe 64 40 0 4 WRAPS wrap 2 0 1 CP
$ /home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/fullrange_conv/fullrange_conv.exe 64 40 0 4 WRAPS wrap 2 2 3 RP
$ /home/d

{('WRAPS',
  'wrap',
  'CP',
  32,
  70): {'averaged': PosixPath('/home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/conv_res/CP_L32_T70_num0_to_4_averaged.txt'), 'convoluted': PosixPath('/home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/conv_res/CP_L32_T70_num0_to_4_convoluted.txt')},
 ('WRAPS',
  'wrap',
  'RP',
  32,
  70): {'averaged': PosixPath('/home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/conv_res/RP_L32_T70_num0_to_4_averaged.txt'), 'convoluted': PosixPath('/home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/conv_res/RP_L32_T70_num0_to_4_convoluted.txt')},
 ('Searches',
  'curve',
  'Searches',
  32,
  70): {'averaged': PosixPath('/home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/conv_res/Searches_L32_T70_num0_to_4_averaged.txt'), 'convoluted': PosixPath('/home/danimuzi/Desktop/RPgit/second_PRX_git/new_git_repo/conv_res/Searches_L32_T70_num0_to_4_convoluted.txt')},
 ('TimePerBond',
  'curve',
  'TimePerBond',
  32,
  70): {'averaged': PosixPa

## Part 3: Display and plots

This section only displays aggregated data with matplotlib.

In [11]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.gridspec import GridSpec

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.labelsize": 11,
    "axes.titlesize": 11,
    "font.size": 10,
    "legend.fontsize": 9,
    "lines.linewidth": 1.6,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
})

def panel_letter(i):
    return chr(ord('a') + i)

def one_row_figure(ncols, width_per_panel=3.6, height=3.0):
    fig = plt.figure(figsize=(width_per_panel * ncols, height), constrained_layout=True)
    gs = GridSpec(1, ncols, figure=fig)
    axes = [fig.add_subplot(gs[0, i]) for i in range(ncols)]
    return fig, axes


In [12]:
# Fitting functions

def powerlaw(x, A, exp):
    return A * x **exp

def affine(x, a, b):
    return a*x+b

def fit_stats(model, x, y, popt):
    residuals = y - model(x, *popt)
    ss_res = np.sum(residuals**2)
    ss_tot = np.sum((y - np.mean(y))**2)
    R2 = 1 - ss_res/ss_tot
    return residuals, R2

In [13]:
def plot_one_row_panels(panel_specs, title=None):
    n = len(panel_specs)
    fig, axes = one_row_figure(n)
    for i, spec in enumerate(panel_specs):
        ax = axes[i]
        for item in spec['datasets']:
            ax.plot(item['x'], item['y'], label=item.get('label'))
        if spec.get('xlabel'):
            ax.set_xlabel(spec['xlabel'])
        if spec.get('ylabel'):
            ax.set_ylabel(spec['ylabel'])
        if spec.get('title'):
            ax.set_title(spec['title'])
        if spec.get('legend', False):
            ax.legend(frameon=False)
    if title:
        fig.suptitle(title)
    return fig, axes

def load_curve(path):
    headers, df = read_aggregated_table(path)
    return headers, df

def save_fig(fig, name):
    out = REPO / 'figures'
    out.mkdir(exist_ok=True)
    fig.savefig(out / name, bbox_inches='tight')


In [ ]:
# Critical exponents and thresholds
pc_rp = 0.6602741
pc_cp = 0.347296355
nu_rp = 1.1694
Df_rp = 1.8423
gamma_rp = 1.928
beta_rp = 0.1844

In [ ]:
# Fig. 1: stages of the rigidity transition
events_path = fullrange_outputs[('Events', 'curve', 'Events', Ls[-1], Ts[-1])]['convoluted']
headers, events_df = load_curve(events_path)
p = events_df.iloc[:, 0].to_numpy(dtype=float)
pivoting = events_df.iloc[:, 2].to_numpy(dtype=float)
rigidification = events_df.iloc[:, 4].to_numpy(dtype=float)
overconstraining = events_df.iloc[:, 6].to_numpy(dtype=float)

fig, axes = one_row_figure(1)
ax = axes[0]
ax.plot(p, pivoting, label='pivoting')
ax.plot(p, rigidification, label='rigidification')
ax.plot(p, overconstraining, label='overconstraining')
ax.axvline(pc_cp, color='k', ls='--', lw=1.0, label='$p_c^{CP}$')
ax.axvline(pc_rp, color='k', ls=':', lw=1.0, label='$p_c^{RP}$')
ax.set_xlabel('bond concentration p')
ax.set_ylabel('event probability')
ax.set_xlim(0.0, 1.0)
ax.legend(frameon=False)
ax.set_title('Stages of the rigidity transition')
save_fig(fig, 'fig1_rp_stages.pdf')
plt.show()

In [ ]:
# Fig. 4: performance of the algorithm
def sample_mean(headers):
    if not headers:
        return None
    vals = headers[0].lstrip('#').split()
    return float(vals[0])

fig, axes = one_row_figure(3, width_per_panel=4.0, height=3.2)

# Panel (a): total time vs system size
ax = axes[0]
Ns = []
times = []
for L, T in zip(Ls, Ts):
    headers, df = read_aggregated_table(aggregation_outputs[('PERFLOG', 'samples', L, T, 0)])
    Ns.append(L * L)
    times.append(sample_mean(headers) / 60.0)
ax.loglog(Ns, times, 'o', label='simulation time')
popt, pcov = curve_fit(powerlaw, Ns, times)
A, exp = popt
xx = np.linspace(Ns[0], Ns[-1], 200)
yy = A * xx**exp
ax.loglog(xx, yy, color='tab:blue', alpha=0.45, lw=1.6, label='$N^{%.2f}$' %exp)
print('Performance scaling exponent: %.2f' %exp)

ax.set_xlabel('system size N')
ax.set_ylabel('time [min]')
ax.set_title('Average time to complete a simulation')
ax.legend(frameon=False)

# Panel (b): pebble searches vs p
ax = axes[1]
for L, T in zip(Ls, Ts):
    path = fullrange_outputs[('Searches', 'curve', 'Searches', L, T)]['convoluted']
    headers, df = load_curve(path)

    ax.plot(
    df.iloc[:, 0].to_numpy(),
    df.iloc[:, 2].to_numpy(),
    label=f'L={L}'
)
    #ax.plot(df.iloc[:, 0], df.iloc[:, 2], label=f'L={L}')
ax.set_xlabel('bond concentration p')
ax.set_ylabel('pebble searches / bond')
ax.set_title('Average number of pebble searches')
ax.legend(frameon=False)

# Panel (c): time per bond vs p
ax = axes[2]
for L, T in zip(Ls, Ts):
    path = fullrange_outputs[('TimePerBond', 'curve', 'TimePerBond', L, T)]['convoluted']
    headers, df = load_curve(path)

    ax.plot(
    df.iloc[:, 0].to_numpy(),
    df.iloc[:, 2].to_numpy(),
    label=f'L={L}'
)

    #ax.plot(df.iloc[:, 0], df.iloc[:, 2], label=f'L={L}')
ax.set_xlabel('bond concentration p')
ax.set_ylabel('time / bond [s]')
ax.set_title('Average time per bond activation')
ax.legend(frameon=False)

save_fig(fig, 'fig4_performance.pdf')
plt.show()


In [ ]:
# Fig. 5: computational complexity scaling

fig, ax = one_row_figure(1, width_per_panel=4.6, height=3.6)
ax = ax[0]
Ns = []
typeI = []
typeII = []
pivots = []
for L, T in zip(Ls, Ts):
    Ns.append(L * L)
    h1, _ = load_aggregated_table(aggregation_outputs[('PERFLOG', 'samples', L, T, 5)])
    h2, _ = load_aggregated_table(aggregation_outputs[('PERFLOG', 'samples', L, T, 6)])
    h3, _ = load_aggregated_table(aggregation_outputs[('PERFLOG', 'samples', L, T, 7)])
    typeI.append(sample_mean(h1))
    typeII.append(sample_mean(h2))
    pivots.append(sample_mean(h3))

Ns = np.asarray(Ns, dtype=float)
typeI = np.asarray(typeI, dtype=float)
typeII = np.asarray(typeII, dtype=float)
pivots = np.asarray(pivots, dtype=float)

series = [
    (typeI, 'o', 'tab:blue', 'type I'),
    (typeII, 's', 'tab:red', 'type II'),
    (pivots, '^', 'tab:green', 'pivots'),
]
for y, marker, color, label in series:
    ax.loglog(Ns, y, marker=marker, color=color, linestyle='None', label=label)
    #slope, amp = power_fit(Ns, y)
    popt, pcov = curve_fit(powerlaw, Ns, y)
    A, exp = popt
    print('%s exponent: %.2f' %(label, exp))
    xx = np.linspace(Ns.min(), Ns.max(), 200)
    yy = A * xx**exp
    ax.loglog(xx, yy, color=color, alpha=0.45, lw=1.6)

ax.set_xlabel('system size N')
ax.set_ylabel('number of operations')
ax.set_title('Computational complexity scaling')
ax.legend(frameon=False)
save_fig(fig, 'fig5_complexity_scaling.pdf')
plt.show()


In [ ]:
# Fig. 6: scaling analysis for RP
def wrapping_moments_from_curve(p, r, derivative=True):
    p = np.asarray(p, dtype=float)
    r = np.asarray(r, dtype=float)
    if derivative:
        prob = np.diff(np.concatenate(([0.0], r)))
    else:
        prob = r.copy()
    prob = np.clip(prob, 0.0, None)
    norm = prob.sum()
    if norm <= 0.0:
        return np.nan, np.nan
    prob /= norm
    pc = float(np.sum(p * prob))
    delta = float(np.sqrt(np.sum(((p - pc) ** 2) * prob)))
    return pc, delta



Ls_arr = np.asarray(Ls, dtype=float)
cp_sxy = []
rp_sxy = []
cp_chi = []
rp_chi = []

wrap_types = [
    ('x', 1, 'tab:blue', 'o', 1.0, 1),
    ('xy', 2, 'tab:orange', 's', 2.0, 2),
    ('e', 3, 'tab:green', '^', 4.0, 3),
    ('1', 4, 'tab:red', 'D', 6.0, 4),
]
wrap_data = {name: {'pc': [], 'delta': []} for name, *_ in wrap_types}

have_fig6_data = all((('CPSxy', 'samples', L, T, 0) in aggregation_outputs and ('RPSxy', 'samples', L, T, 0) in aggregation_outputs and ('WRAPS', 'samples', L, T, 4) in aggregation_outputs and ('WRAPS', 'samples', L, T, 5) in aggregation_outputs and ('WRAPS', 'wrap', 'RP', L, T) in fullrange_outputs) for L, T in zip(Ls, Ts))
if have_fig6_data:
    for L, T in zip(Ls, Ts):
        h_cp_sxy, _ = load_aggregated_table(aggregation_outputs[('CPSxy', 'samples', L, T, 0)])
        h_rp_sxy, _ = load_aggregated_table(aggregation_outputs[('RPSxy', 'samples', L, T, 0)])
        h_cp_chi, _ = load_aggregated_table(aggregation_outputs[('WRAPS', 'samples', L, T, 4)])
        h_rp_chi, _ = load_aggregated_table(aggregation_outputs[('WRAPS', 'samples', L, T, 5)])
        cp_sxy.append(sample_mean(h_cp_sxy) / (3.0 * L * L))
        rp_sxy.append(sample_mean(h_rp_sxy) / (3.0 * L * L))
        cp_chi.append(sample_mean(h_cp_chi) / (3.0 * L * L))
        rp_chi.append(sample_mean(h_rp_chi) / (3.0 * L * L))

        headers, df = load_curve(fullrange_outputs[('WRAPS', 'wrap', 'RP', L, T)]['convoluted'])
        p = df.iloc[:, 0].to_numpy(dtype=float)
        curves = {
            'x': df.iloc[:, 1].to_numpy(dtype=float),
            'xy': df.iloc[:, 2].to_numpy(dtype=float),
            'e': df.iloc[:, 3].to_numpy(dtype=float),
            '1': df.iloc[:, 4].to_numpy(dtype=float),
        }
        for name, _, _, _, shift, _ in wrap_types:
            pc, delta = wrapping_moments_from_curve(p, curves[name], derivative=(name != '1'))
            wrap_data[name]['pc'].append(pc)
            wrap_data[name]['delta'].append(delta)
else:
    print('Skipping Fig. 6: WRAPS/CPSxy/RPSxy outputs are not available yet.')

cp_sxy = np.asarray(cp_sxy, dtype=float)
rp_sxy = np.asarray(rp_sxy, dtype=float)
cp_chi = np.asarray(cp_chi, dtype=float)
rp_chi = np.asarray(rp_chi, dtype=float)

fig, axes = one_row_figure(4, width_per_panel=3.5, height=3.2)

# Panel (a): wrapping-cluster size
ax = axes[0]
ax.loglog(Ls_arr, cp_sxy, 'o', color='tab:blue', label='CP')
ax.loglog(Ls_arr, rp_sxy, 's', color='tab:orange', label='RP')
for y, color, start, model in [(cp_sxy, 'tab:blue', 0, 'CP'), (rp_sxy, 'tab:orange', 0, 'RP')]:
    popt, pcov = curve_fit(powerlaw, Ls_arr[start:], y[start:])
    A, exp = popt
    print('%s: D_f = %.6f' %(model, exp+2))
    xx = np.linspace(Ls_arr.min(), Ls_arr.max(), 200)
    ax.loglog(xx, A * xx**exp, color=color, alpha=0.45, lw=1.6)
ax.set_xlabel('L')
ax.set_ylabel(r'$S_{xy}/(3L^2)$')
ax.set_title('Wrapping-cluster size')
ax.legend(frameon=False)

print('\n')

# Panel (b): maximum susceptibility
ax = axes[1]
ax.loglog(Ls_arr, cp_chi, 'o', color='tab:blue', label='CP')
ax.loglog(Ls_arr, rp_chi, 's', color='tab:orange', label='RP')
for y, color, start, model in [(cp_chi, 'tab:blue', 0, 'CP'), (rp_chi, 'tab:orange', 0, 'RP')]:
    popt, pcov = curve_fit(powerlaw, Ls_arr[start:], y[start:])
    A, exp = popt
    print('%s: gamma/nu = %.6f' %(model, exp+2))
    xx = np.linspace(Ls_arr.min(), Ls_arr.max(), 200)
    ax.loglog(xx, A * xx**exp, color=color, alpha=0.45, lw=1.6)
ax.set_xlabel('L')
ax.set_ylabel(r'$\chi_{\max}/(3L^2)$')
ax.set_title('Maximum susceptibility')
ax.legend(frameon=False)

print('\n')
# Panel (c): transition widths from wrapping probabilities
ax = axes[2]
for name, _, color, marker, shift, label_shift in wrap_types:
    delta = np.asarray(wrap_data[name]['delta'], dtype=float) * shift
    this_label = f'R{name}'
    ax.loglog(Ls_arr, delta, marker=marker, color=color, linestyle='None', label=this_label)
    popt, pcov = curve_fit(powerlaw, Ls_arr[start:], delta[start:])
    A, exp = popt
    print('%s: 1/nu = %.6f' %(this_label, -exp))
    xx = np.linspace(Ls_arr.min(), Ls_arr.max(), 200)
    ax.loglog(xx, A * xx**exp, color=color, alpha=0.45, lw=1.6)
ax.set_xlabel('L')
ax.set_ylabel(r'$\Delta$')
ax.set_title('Wrapping widths')
ax.legend(frameon=False)

print('\n')
# Panel (d): finite-size thresholds
ax = axes[3]
start=0
for name, _, color, marker, shift, label_shift in wrap_types:
    delta = np.asarray(wrap_data[name]['delta'], dtype=float)
    pc = np.asarray(wrap_data[name]['pc'], dtype=float)
    x = 1.0 / delta
    y = (pc / delta)
    popt, pcov = curve_fit(affine, x[start:], y[start:])
    a, b = popt
    xx = np.linspace(x.min(), x.max(), 200)
    this_label = f'R{name}'
    ax.loglog(xx, affine(xx,a,b) * shift, color=color, alpha=0.45, lw=1.6)
    ax.loglog(x, y * shift, marker=marker, color=color, linestyle='None', label=this_label)
    print('%s: p_c = %.6f' %(this_label, -exp))

ax.set_xlabel(r'$\Delta^{-1}$')
ax.set_ylabel(r'$p_c/\Delta$')
ax.set_title('Thresholds')
ax.legend(frameon=False)

save_fig(fig, 'fig6_exponents.pdf')
plt.show()


In [ ]:
# Fig. 7: scaling-function collapse for RP
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

beta_over_nu = beta_rp / nu_rp
gamma_over_nu = gamma_rp / nu_rp

fig, axes = one_row_figure(3, width_per_panel=4.0, height=3.4)
colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(Ls)))

def collapse_x(p, L):
    return (np.asarray(p, dtype=float) - pc_rp) * (L ** (1.0 / nu_rp))

def load_partialrange_convoluted(path, L):
    headers, df = load_curve(path)
    if not headers:
        raise RuntimeError(f'Missing header in convoluted file: {path}')
    parts = headers[0].lstrip('#').split()
    if len(parts) < 1:
        raise RuntimeError(f'Invalid convoluted header in {path}: {headers[0]}')
    p0 = float(parts[0])
    dp = 1.0 / float(3 * L * L)
    p = p0 + dp * np.arange(len(df), dtype=float)
    return p, df

def plot_collapse_panel(ax, inset_loc, ylabel, yscale, data_key, raw_label, inset_title):
    inset = inset_axes(ax, width='36%', height='36%', loc=inset_loc, borderpad=1.0)
    for color, L, T in zip(colors, Ls, Ts):
        if data_key == 'Pinf':
            key = ('ROP', L, T)
            if key not in partialrange_outputs:
                continue
            p, df = load_partialrange_convoluted(partialrange_outputs[key]['convoluted'], L)
            y = df.iloc[:, 0].to_numpy(dtype=float)
            y_main = y
            y_inset = y * (L ** beta_over_nu)
        elif data_key == 'Chi':
            key = ('ROP', L, T)
            if key not in partialrange_outputs:
                continue
            p, df = load_partialrange_convoluted(partialrange_outputs[key]['convoluted'], L)
            y = df.iloc[:, 1].to_numpy(dtype=float)
            y_main = y
            y_inset = y * (L ** (-gamma_over_nu))
        else:
            key = ('WRAPS', 'wrap', 'RP', L, T)
            if key not in fullrange_outputs:
                continue
            headers, df = load_curve(fullrange_outputs[key]['convoluted'])
            p = df.iloc[:, 0].to_numpy(dtype=float)
            y = df.iloc[:, 2].to_numpy(dtype=float)
            y_main = y
            y_inset = y
        ax.plot(p, y_main, color=color, lw=1.3, alpha=0.85)
        x = collapse_x(p, L)
        inset.plot(x, y_inset, color=color, lw=1.0, alpha=0.9)
    ax.set_xlabel('p')
    ax.set_ylabel(ylabel)
    ax.set_title(raw_label)
    inset.set_title(inset_title, fontsize=8)
    inset.yaxis.tick_right()
    inset.yaxis.set_label_position('right')
    inset.tick_params(axis='y', right=True, labelright=True, left=False, labelleft=False)
    inset.spines['left'].set_visible(False)
    inset.spines['right'].set_visible(True)
    return inset

ax = axes[0]
plot_collapse_panel(ax, 'upper left', r'$P_{\infty}$', 1.0, 'Pinf', r'Order parameter $P_{\infty}$', 'collapse')
ax.axvline(pc_rp, color='k', ls='--', lw=1.0)

ax = axes[1]
plot_collapse_panel(ax, 'upper left', r'$\chi$', 1.0, 'Chi', r'Susceptibility $\chi$', 'collapse')
ax.axvline(pc_rp, color='k', ls='--', lw=1.0)

ax = axes[2]
plot_collapse_panel(ax, 'upper left', r'$R_{xy}$', 1.0, 'Rxy', r'Wrapping probability $R_{xy}$', 'collapse')
ax.axvline(pc_rp, color='k', ls='--', lw=1.0)

save_fig(fig, 'fig7_collapse.pdf')
plt.show()


In [ ]:
# Fig. S1: scaling of the computing time of each event class
fig, ax = one_row_figure(1, width_per_panel=4.8, height=3.6)
ax = ax[0]

Ns = []
pivoting_time = []
rigidification_time = []
overconstraining_time = []

for L, T in zip(Ls, Ts):
    Ns.append(L * L)
    h_piv, _ = load_aggregated_table(aggregation_outputs[('PERFLOG', 'samples', L, T, 8)])
    h_rig, _ = load_aggregated_table(aggregation_outputs[('PERFLOG', 'samples', L, T, 9)])
    h_ovr, _ = load_aggregated_table(aggregation_outputs[('PERFLOG', 'samples', L, T, 10)])
    pivoting_time.append(sample_mean(h_piv))
    rigidification_time.append(sample_mean(h_rig))
    overconstraining_time.append(sample_mean(h_ovr))

Ns = np.asarray(Ns, dtype=float)
pivoting_time = np.asarray(pivoting_time, dtype=float)
rigidification_time = np.asarray(rigidification_time, dtype=float)
overconstraining_time = np.asarray(overconstraining_time, dtype=float)

series = [
    (pivoting_time, 'o', 'tab:blue', 'pivoting'),
    (rigidification_time, 's', 'tab:red', 'rigidification'),
    (overconstraining_time, '^', 'tab:green', 'overconstraining'),
]

for y, marker, color, label in series:
    ax.loglog(Ns, y, marker=marker, color=color, linestyle='None', label=label)
    popt, pcov = curve_fit(powerlaw, Ns, y)
    A, exp = popt
    print('%s scales with exponent %.2f' %(label, exp))
    xx = np.linspace(Ns.min(), Ns.max(), 200)
    yy = A * xx**exp
    ax.loglog(xx, yy, color=color, alpha=0.45, lw=1.6)

ax.set_xlabel('system size N')
ax.set_ylabel('computing time')
ax.set_title('Scaling of event cost')
ax.legend(frameon=False)
save_fig(fig, 'figS1_event_cost.pdf')
plt.show()


In [ ]:
# Fig. S4: average number of operations as a function of the bond concentration
fig, axes = one_row_figure(3, width_per_panel=4.0, height=3.4)

panel_specs = [
    ('type I', 0),
    ('type II', 1),
    ('pivots', 2),
]
line_colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(Ls)))

for i, ((title, col_id), ax) in enumerate(zip(panel_specs, axes)):
    for color, L, T in zip(line_colors, Ls, Ts):
        path = fullrange_outputs[('Operations', 'curve', 'Operations', L, T)]['convoluted']
        headers, df = load_curve(path)
        p = df.iloc[:, 0].to_numpy(dtype=float)
        y = df.iloc[:, 2 * col_id + 2].to_numpy(dtype=float)
        ax.plot(p, y, color=color, lw=1.2, alpha=0.9, label=f'L={L}')
    ax.set_xlabel('p')
    if i == 0:
        ax.set_ylabel('number of operations')
    ax.set_title(title)
    ax.legend(frameon=False)

save_fig(fig, 'figS4_comp_complex_curves.pdf')
plt.show()

In [ ]:
# Fig. S6: wrapping probabilities and their collapse

wrap_labels = [r'$R_x$', r'$R_{xy}$', r'$R_e$', r'$R_1$']
wrap_cols = [1, 2, 3, 4]
colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']

fig, axes = plt.subplots(2, 4, figsize=(15.0, 6.4), constrained_layout=True)

have_s6 = all((('WRAPS', 'wrap', 'RP', L, T) in fullrange_outputs) for L, T in zip(Ls, Ts))
if have_s6:
    for j, (lab, col, color) in enumerate(zip(wrap_labels, wrap_cols, colors)):
        ax = axes[0, j]
        for L, T in zip(Ls, Ts):
            headers, df = load_curve(fullrange_outputs[('WRAPS', 'wrap', 'RP', L, T)]['convoluted'])
            p = df.iloc[:, 0].to_numpy(dtype=float)
            y = df.iloc[:, col].to_numpy(dtype=float)
            ax.plot(p, y, color=color, lw=1.2, alpha=0.9)
        ax.axvline(pc_rp, color='k', ls='--', lw=1.0)
        ax.set_title(lab)
        ax.set_xlabel('p')
        if j == 0:
            ax.set_ylabel('wrapping probability')

    for j, (lab, col, color) in enumerate(zip(wrap_labels, wrap_cols, colors)):
        ax = axes[1, j]
        for L, T in zip(Ls, Ts):
            headers, df = load_curve(fullrange_outputs[('WRAPS', 'wrap', 'RP', L, T)]['convoluted'])
            p = df.iloc[:, 0].to_numpy(dtype=float)
            y = df.iloc[:, col].to_numpy(dtype=float)
            x = (p - pc_rp) * (L ** (1.0 / nu_rp))
            ax.plot(x, y, color=color, lw=1.1, alpha=0.85)
        ax.set_xlabel(r'$(p-p_c^{RP})L^{1/\nu^{RP}}$')
        if j == 0:
            ax.set_ylabel('wrapping probability')
else:
    print('Skipping Fig. S6: WRAPS convolutions are not available yet.')

save_fig(fig, 'figS6_wrapping_probs.pdf')
plt.show()

